In [1]:
import pandas as pd

In [20]:
# import kagglehub
# # Download latest version
# path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
# print("Path to dataset files:", path)

In [21]:
df = pd.read_csv("IMDB Dataset.csv")
df.sample(5)

,review,sentiment
34468,The arrival of White Men in Arctic Canada chal...,positive
24047,I can't say this is the worst film of all time...,negative
49637,If you liked the Richard Chamberlain version o...,positive
36260,Probably because this is Columbia's first film...,positive
15427,Okay. Look- I've seen LOTS and I do mean LOTS ...,negative


In [22]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [23]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [24]:
df['sentiment'] =df['sentiment'].apply(lambda x: 0 if x=='positive' else 1)
df.sample(5)

,review,sentiment
43291,The Lion King 1 1/2 is a very cute story to go...,0
28005,"After being bitten by a bat in a cave,a doctor...",1
949,The film someone had to make.<br /><br />Waco:...,0
17125,"This is superb - the acting wonderful, sets, c...",0
14367,"Some films just fade away, but Tourist Trap ha...",0


In [25]:
from  sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(df.review ,df.sentiment,test_size=0.20)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)

(40000,)
(10000,)
(40000,)


In [26]:
X_train.values[0:2]

array(['Totally forgettable movie but an unbelievable soundtrack: I\'d give it (soundtrack)a 9 out of 10. I have the CD and the guitar work (Nils Lofgren) is superb! I saw the movie years ago and had to check IMDb to remember what it was about. I obsessed about getting the soundtrack and have since had to replace it. It ranges from blues/soul/ballad to a dose of gospel. All songs written, arranged, produced and performed by Nils Lofgren who is the "other" lead guitarist opposite Steve Van Zandt in the E Street Band. This dude can play! The vocals are handled by Nils (he can\'t sing very good-too raspy), Bonnie Sheridan (who is a great singer) and Tom Lepson.',
       'Coinciding with the start of the baby boom, the years after World War II saw an unprecedented exodus of Americans moving out of their city apartments into the suburbs where they can fulfill their dreams of owning their own homes. Directed by H.C. Potter and co-written by Norman Panama and Melvin Frank ("White Christmas"),

In [27]:
type(X_train)

pandas.core.series.Series

In [28]:
X_train

23541    Totally forgettable movie but an unbelievable ...
34414    Coinciding with the start of the baby boom, th...
26908    This movie which was released directly on vide...
7878     Bad sequels.....this one's a real one! When th...
4905     I fell asleep on my couch at 7:35pm last night...
                               ...                        
20626    Absolutely amazing! Humor, up-beat music and a...
35289    Watching this again recently, I found it heart...
49254    As interesting as a sheet of cardboard, this d...
4848     Ok first of all, this movie sucks. But lets ex...
18490    I was surprised how much I enjoyed this. Sure ...
Name: review, Length: 40000, dtype: object

### **Bag of words : -**

In [29]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer()

x_train_cv = v.fit_transform(X_train.values)
x_train_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5460248 stored elements and shape (40000, 92744)>

In [30]:
x_train = x_train_cv.toarray()

In [31]:
x_train[0:3]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(3, 92744))

In [32]:
v.get_feature_names_out()

array(['00', '000', '00000000000', ..., 'ýs', 'þór', 'żmijewski'],
      shape=(92744,), dtype=object)

In [33]:
X_train.values

array(['Totally forgettable movie but an unbelievable soundtrack: I\'d give it (soundtrack)a 9 out of 10. I have the CD and the guitar work (Nils Lofgren) is superb! I saw the movie years ago and had to check IMDb to remember what it was about. I obsessed about getting the soundtrack and have since had to replace it. It ranges from blues/soul/ballad to a dose of gospel. All songs written, arranged, produced and performed by Nils Lofgren who is the "other" lead guitarist opposite Steve Van Zandt in the E Street Band. This dude can play! The vocals are handled by Nils (he can\'t sing very good-too raspy), Bonnie Sheridan (who is a great singer) and Tom Lepson.',
       'Coinciding with the start of the baby boom, the years after World War II saw an unprecedented exodus of Americans moving out of their city apartments into the suburbs where they can fulfill their dreams of owning their own homes. Directed by H.C. Potter and co-written by Norman Panama and Melvin Frank ("White Christmas"),

In [35]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB

# Define stratified CV with random shuffling
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Hyperparameters to tune
param_grid = {
    "alpha": [0.01, 0.8, 0.5, 0.6,0.7,0.9,1, 2, 5],
    "fit_prior": [True, False]
}

# Model
nb = MultinomialNB()

# GridSearchCV
grid = GridSearchCV(
    estimator=nb,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

# Fit
grid.fit(x_train_cv, y_train)

# Results
print("Best Parameters:", grid.best_params_)
print("Best F1 Score:", grid.best_score_)


Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best Parameters: {'alpha': 0.8, 'fit_prior': True}
Best F1 Score: 0.8519601831188846


In [39]:
from sklearn.naive_bayes import MultinomialNB

# 🔹 Put YOUR parameters here
model = MultinomialNB(
    alpha=0.8,        
    fit_prior=False    
)

# 🔹 Train the model
model.fit(x_train_cv, y_train)



,alpha,0.8
,force_alpha,True
,fit_prior,False
,class_prior,None


In [40]:
x_test_cv = v.transform(X_test)
x_test_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1356528 stored elements and shape (10000, 92744)>

In [41]:
x_test = x_test_cv.toarray()
x_test

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(10000, 92744))

In [42]:
from sklearn.metrics import classification_report
y_pred = model.predict(x_test_cv)
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

           0       0.88      0.82      0.85      5028
           1       0.83      0.88      0.86      4972

    accuracy                           0.85     10000
   macro avg       0.86      0.85      0.85     10000
weighted avg       0.86      0.85      0.85     10000

